# Setup & Data Preparation تجهيز الداتا

In [ ]:
# Import PyTorch and neural network utilities
import torch
import torch.nn as nn
from torch.nn import functional as F

# Use GPU if available, otherwise use CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Display the selected device
print("Using:", device)

Using: cpu


In [ ]:
# Install a version that supports dataset loading scripts
!pip install -q "datasets==3.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 16.2 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "inparallel/saudinewsnet",
    split="train",
    trust_remote_code=True
)

print(dataset)
print(dataset[0])

README.md:   0%|          | 0.00/8.26k [00:00<?, ?B/s]

saudinewsnet.py:   0%|          | 0.00/5.87k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31030 [00:00<?, ? examples/s]

Dataset({
    features: ['source', 'url', 'date_extracted', 'title', 'author', 'content'],
    num_rows: 31030
})
{'source': 'aawsat', 'url': 'http://aawsat.com/home/article/410826/بريطانيا-أربعة-محاور-لاستراتيجية-جديدة-تتصدى-للتطرف-على-مدى-خمس-سنوات', 'date_extracted': '2015-07-21 02:51:32', 'title': 'بريطانيا: أربعة محاور لاستراتيجية جديدة تتصدى للتطرف على مدى خمس سنوات', 'author': 'لندن: رنيم حنوش', 'content': 'حدد رئيس الوزراء البريطاني ديفيد كاميرون، اليوم (الاثنين)، ملامح استراتيجية للتصدي للتطرف داخل بريطانيا؛ وهي مسألة اعتبرها كاميرون "صراع جيلنا"، متعهدا خلال خطابه في مدينة بيرمنغهام بالتصدي لهؤلاء الذين ينشرون التطرف بين الشبان المسلمين البريطانيين.\n\n ورسم كاميرون الاطار العام لاستراتيجية مكافحة التطرف التي المقرر ان تنشر كاملة في وقت لاحق هذا العام، والتي تسعى للتصدي لانتشار الأفكار المتطرفة التي يروج لها متشددو تنظيم "داعش".\n\n وحسبما تناقلت وسائل الإعلام البريطانية، فإن خطة رئيس الوزراء ستكون على مدى خمسة أعوام للقضاء على التطرف الداخلي من خلال أربعة محاور، وهي: القضاء 

In [ ]:
# Shuffle the dataset so we get articles from different sources
dataset = dataset.shuffle(seed=42)

# Use only 5,000 articles to keep training manageable
small_dataset = dataset.select(range(5000))

# Store all formatted news articles in one text
articles = []

# Go through every article in the selected dataset
for article in small_dataset:

    # Get the article title and content
    title = article["title"].strip()
    content = article["content"].strip()

    # Skip articles that have an empty title or content
    if not title or not content:
        continue

    # Format each article in a consistent structure
    formatted_article = (
        f"العنوان: {title}\n"
        f"الخبر: {content}\n\n"
    )

    # Add the formatted article to the list
    articles.append(formatted_article)

# Combine all articles into one large text
text = "".join(articles)

# Display basic information about our prepared dataset
print("Number of articles:", len(articles))
print("Number of characters:", len(text))

# Display a sample
print("\nSample:\n")
print(text[:1000])

Number of articles: 4952
Number of characters: 8805112

Sample:

العنوان: الإمارات الأولى عالمياً في مؤشر التماسك الاجتماعي
الخبر: احتلت دولة الإمارات المرتبة الأولى عالمياً في مؤشر التماسك الاجتماعي الصادر عن المعهد الدولي للتنمية الإدارية، الذي أشاد بدور التماسك الاجتماعي في التنافسية والازدهار الاقتصادي طويل المدى. وقال المعهد إن تصدر الإمارات المرتبة الأولى عالمياً في التماسك الاجتماعي يعكس مدى ما تتمتع به من استقرار سياسي، ويوفر لها التركيز على النمو الاقتصادي وتطوير قطاعات الأعمال، لافتاً إلى أن التماسك الاجتماعي هو ترجمة حقيقية لمسيرة من الجهد والعطاء يمتد بناؤها لسنوات طويلة ولا تأتي بين عشية أو ضحاها. وأكد البروفيسور ارتو بريس مدير مركز التنافسية الدولي بالمعهد، أن التماسك الاجتماعي هو نتاج السياسات المثمرة التي تتيح لكل فرد في المجتمع المشاركة في رخائه، فضلاً عن كونه يوفر عاملاً رئيسياً لاستدامة التنافسية. وأضاف بريس أن تصنيف المعهد الدولي للتنمية الإدارية في تقرير الكتاب السنوي للتنافسية 2013 يعكس أهمية مؤشر التماسك الاجتماعي بالنسبة لرخاء الدول، فالدول المتصدرة للمؤشر وهي ا

In [ ]:
# Import tools for Unicode text normalization
import unicodedata

# Normalize Unicode characters
# NFKC converts compatibility forms such as Arabic presentation forms
# into their standard character representation
text = unicodedata.normalize("NFKC", text)

# Remove invisible control and formatting characters
# while keeping normal new lines
cleaned_chars = []

for char in text:

    # Get the Unicode category of the character
    category = unicodedata.category(char)

    # Keep normal characters and new lines
    if not category.startswith("C") or char == "\n":
        cleaned_chars.append(char)

# Join the cleaned characters back into one text
text = "".join(cleaned_chars)

# Display the cleaned dataset size
print("Number of characters after cleaning:", len(text))

# Display a sample after cleaning
print("\nCleaned sample:\n")
print(text[:1000])

Number of characters after cleaning: 8804939

Cleaned sample:

العنوان: الإمارات الأولى عالمياً في مؤشر التماسك الاجتماعي
الخبر: احتلت دولة الإمارات المرتبة الأولى عالمياً في مؤشر التماسك الاجتماعي الصادر عن المعهد الدولي للتنمية الإدارية، الذي أشاد بدور التماسك الاجتماعي في التنافسية والازدهار الاقتصادي طويل المدى. وقال المعهد إن تصدر الإمارات المرتبة الأولى عالمياً في التماسك الاجتماعي يعكس مدى ما تتمتع به من استقرار سياسي، ويوفر لها التركيز على النمو الاقتصادي وتطوير قطاعات الأعمال، لافتاً إلى أن التماسك الاجتماعي هو ترجمة حقيقية لمسيرة من الجهد والعطاء يمتد بناؤها لسنوات طويلة ولا تأتي بين عشية أو ضحاها. وأكد البروفيسور ارتو بريس مدير مركز التنافسية الدولي بالمعهد، أن التماسك الاجتماعي هو نتاج السياسات المثمرة التي تتيح لكل فرد في المجتمع المشاركة في رخائه، فضلاً عن كونه يوفر عاملاً رئيسياً لاستدامة التنافسية. وأضاف بريس أن تصنيف المعهد الدولي للتنمية الإدارية في تقرير الكتاب السنوي للتنافسية 2013 يعكس أهمية مؤشر التماسك الاجتماعي بالنسبة لرخاء الدول، فالدول المتصدرة للمؤشر وهي الإ

In [ ]:
# Extract all unique characters from the cleaned Arabic news text
chars = sorted(list(set(text)))

# Count the number of unique characters
vocab_size = len(chars)

# Display the vocabulary size
print("Vocabulary size:", vocab_size)

# Display all unique characters
print("Characters:")
print(chars)

Vocabulary size: 184
Characters:
['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '«', '·', '»', 'Ä', 'È', '×', 'á', 'ã', 'é', 'ê', 'ü', 'š', '،', '؛', '؟', 'ء', 'آ', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ـ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', 'ً', 'ٌ', 'ٍ', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', 'ٓ', '٠', '١', '٢', '٣', '٤', '٥', '٦', '٧', '٨', '٩', '٪', '٫', '٬', 'پ', 'چ', 'ژ', 'ڤ', 'گ', 'ی', '–', '—', '’', '“', '”', '•', '′', '›', '■', '▪', '●', '﴾', '﴿']


In [ ]:
# Create a mapping from each character to a unique integer ID
stoi = {ch: i for i, ch in enumerate(chars)}

# Create the reverse mapping from integer IDs back to characters
itos = {i: ch for i, ch in enumerate(chars)}

# Convert text into a list of integer IDs
def encode(s):
    return [stoi[c] for c in s]

# Convert integer IDs back into readable text
def decode(ids):
    return ''.join([itos[i] for i in ids])

In [ ]:
# Test the encoder and decoder with Arabic text
test_text = "الخبر"

# Encode the text into integer IDs
encoded = encode(test_text)

# Decode the IDs back into readable text
decoded = decode(encoded)

print("Original:", test_text)
print("Encoded :", encoded)
print("Decoded :", decoded)

Original: الخبر
Encoded : [112, 136, 119, 113, 122]
Decoded : الخبر


In [ ]:
# Encode the full Arabic news text
# and convert it into a PyTorch tensor
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

# Display the tensor shape
print("Data shape:", data.shape)

# Display the first 20 token IDs
print("First 20 tokens:", data[:20])

Data shape: torch.Size([8804939])
First 20 tokens: tensor([112, 136, 130, 138, 140, 112, 138,  27,   1, 112, 136, 110, 137, 112,
        122, 112, 115,   1, 112, 136])


In [ ]:
# Split the dataset into 90% training data
# and 10% validation data
n = int(0.9 * len(data))

# First 90% for training
train_data = data[:n]

# Remaining 10% for validation
val_data = data[n:]

# Display the size of each split
print("Training:", len(train_data))
print("Validation:", len(val_data))

Training: 7924445
Validation: 880494


In [ ]:
# Define a small sequence length
# This is only for understanding how next-character prediction works
block_size = 8

# Take the first 8 characters from the training data
x = train_data[:block_size]

# Create the target sequence by shifting one character forward
y = train_data[1:block_size + 1]

# Display the input sequence
print("Input:")
print(decode(x.tolist()))

# Display the target sequence
print("\nTarget:")
print(decode(y.tolist()))

Input:
العنوان:

Target:
لعنوان: 


In [ ]:
# Define how many sequences are processed together
batch_size = 32

# Define the length of each training sequence
block_size = 128


# Create a batch of input and target sequences
def get_batch(split):

    # Select training or validation data
    data_source = train_data if split == 'train' else val_data

    # Randomly choose starting positions for each sequence
    ix = torch.randint(
        len(data_source) - block_size,
        (batch_size,)
    )

    # Create input sequences
    x = torch.stack([
        data_source[i:i + block_size]
        for i in ix
    ])

    # Create target sequences shifted by one character
    y = torch.stack([
        data_source[i + 1:i + block_size + 1]
        for i in ix
    ])

    # Move the batch to GPU or CPU
    x = x.to(device)
    y = y.to(device)

    return x, y

In [ ]:
# Get one training batch
xb, yb = get_batch('train')

# Display the shapes
print("Input shape :", xb.shape)
print("Target shape:", yb.shape)

Input shape : torch.Size([32, 128])
Target shape: torch.Size([32, 128])


# Building the Transformer Model بناء النموذج

In [48]:
# Size of the embedding vector for each character
n_embd = 128

# Number of self-attention heads
n_head = 4

# Number of Transformer blocks
n_layer = 4

# Dropout helps reduce overfitting during training
dropout = 0.1

# Display the model configuration
print("Embedding size:", n_embd)
print("Attention heads:", n_head)
print("Transformer blocks:", n_layer)
print("Dropout:", dropout)

Embedding size: 128
Attention heads: 4
Transformer blocks: 4
Dropout: 0.1


In [49]:
class Head(nn.Module):
    """One head of self-attention"""

    def __init__(self, head_size):
        super().__init__()

        # Create Key, Query, and Value projections
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Create a causal mask
        # This prevents the model from looking at future characters
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        # Apply dropout to reduce overfitting
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # B = batch size
        # T = sequence length
        # C = embedding dimension
        B, T, C = x.shape

        # Create Key and Query representations
        k = self.key(x)
        q = self.query(x)

        # Calculate attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5

        # Prevent attention to future characters
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        # Convert attention scores into probabilities
        wei = F.softmax(wei, dim=-1)

        # Apply dropout
        wei = self.dropout(wei)

        # Create Value representations
        v = self.value(x)

        # Combine the values based on attention weights
        out = wei @ v

        return out

In [50]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        # Create multiple self-attention heads
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        # Combine the outputs of all heads
        # and project them back to the embedding size
        self.proj = nn.Linear(
            head_size * num_heads,
            n_embd
        )

        # Apply dropout to reduce overfitting
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Run all attention heads in parallel
        # then combine their outputs
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        # Project the combined output
        out = self.dropout(self.proj(out))

        return out

In [51]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        # Feed-forward neural network
        self.net = nn.Sequential(

            # Expand the embedding dimension
            nn.Linear(n_embd, 4 * n_embd),

            # Apply a nonlinear activation function
            nn.GELU(),

            # Project back to the original embedding size
            nn.Linear(4 * n_embd, n_embd),

            # Apply dropout to reduce overfitting
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [52]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        # Calculate the size of each attention head
        head_size = n_embd // n_head

        # Multi-head self-attention
        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        # Feed-forward neural network
        self.ffwd = FeedForward(n_embd)

        # Layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Self-Attention + Residual Connection
        x = x + self.sa(self.ln1(x))

        # FeedForward + Residual Connection
        x = x + self.ffwd(self.ln2(x))

        return x

# MiniGPT or NanoGPT Model Architecture هيكل النموذج

In [53]:
class MiniGPT(nn.Module):

    def __init__(self):
        super().__init__()

        # Convert token IDs into dense embedding vectors
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Learn the position of each token in the sequence
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Stack multiple Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Convert hidden representations into vocabulary predictions
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Get token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Get positional embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        # Combine token meaning with token position
        x = tok_emb + pos_emb

        # Pass through Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Produce a score for every possible next token
        logits = self.lm_head(x)

        loss = None

        # Calculate training loss if target tokens are provided
        if targets is not None:

            B, T, C = logits.shape

            logits_flat = logits.reshape(B * T, C)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits_flat,
                targets_flat
            )

        return logits, loss

    def generate(
        self,
        idx,
        max_new_tokens,
        temperature=0.7,
        top_k=20
    ):

        # Generate one character at a time
        for _ in range(max_new_tokens):

            # Keep only the latest context window
            idx_cond = idx[:, -block_size:]

            # Get model predictions
            logits, _ = self(idx_cond)

            # Use only the prediction for the last position
            logits = logits[:, -1, :]

            # Control randomness
            logits = logits / temperature

            # Keep only the top-k most likely characters
            if top_k is not None:

                values, _ = torch.topk(
                    logits,
                    min(top_k, logits.size(-1))
                )

                logits[
                    logits < values[:, [-1]]
                ] = float("-inf")

            # Convert scores into probabilities
            probs = F.softmax(
                logits,
                dim=-1
            )

            # Sample the next character
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add the predicted character
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [54]:
# Create a fresh V2 model
model = MiniGPT().to(device)

# Create a new optimizer for V2
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

# Train for up to 6000 iterations
max_iters = 6000

# Evaluate every 200 iterations
eval_interval = 200

print("V2 model ready!")
print("Max iterations:", max_iters)
print("Evaluation interval:", eval_interval)

V2 model ready!
Max iterations: 6000
Evaluation interval: 200


In [55]:
# Check whether the model has a generate() method
print(hasattr(model, "generate"))

True


# Training the Model تدريب النموذج    

In [43]:
# Create a new optimizer for the currently loaded model
# Use a smaller learning rate for continued training
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

# Number of additional training iterations
additional_iters = 2000

# Evaluate every 200 steps
eval_interval = 200

print("Optimizer ready for continued training!")

Optimizer ready for continued training!


In [56]:
# Disable gradient calculation during evaluation
@torch.no_grad()
def estimate_loss():

    # Switch the model to evaluation mode
    model.eval()

    results = {}

    # Evaluate both training and validation data
    for split in ["train", "val"]:

        # Store loss values from multiple batches
        losses = torch.zeros(20)

        # Calculate the loss over 20 batches
        for k in range(20):

            # Get a batch from training or validation data
            X, Y = get_batch(split)

            # Make predictions and calculate the loss
            logits, loss = model(X, Y)

            # Store the loss value
            losses[k] = loss.item()

        # Calculate the average loss
        results[split] = losses.mean().item()

    # Switch the model back to training mode
    model.train()

    return results

In [57]:
# Check the current loss before continuing training
losses = estimate_loss()

print(
    f"Current Train Loss = {losses['train']:.4f}, "
    f"Current Val Loss = {losses['val']:.4f}"
)

Current Train Loss = 5.4030, Current Val Loss = 5.4046


In [58]:
# Continue training the existing model
for step in range(additional_iters):

    # Evaluate the model every 200 steps
    if step % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Additional Step {step}: "
            f"Train Loss = {losses['train']:.4f}, "
            f"Val Loss = {losses['val']:.4f}"
        )

    # Get a training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Clear previous gradients
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Update model weights
    optimizer.step()

Additional Step 0: Train Loss = 5.4016, Val Loss = 5.4042
Additional Step 200: Train Loss = 2.7677, Val Loss = 2.7625
Additional Step 400: Train Loss = 2.6836, Val Loss = 2.6691
Additional Step 600: Train Loss = 2.6293, Val Loss = 2.6213
Additional Step 800: Train Loss = 2.5346, Val Loss = 2.5286
Additional Step 1000: Train Loss = 2.4659, Val Loss = 2.4450
Additional Step 1200: Train Loss = 2.3988, Val Loss = 2.3841
Additional Step 1400: Train Loss = 2.3447, Val Loss = 2.3489
Additional Step 1600: Train Loss = 2.3027, Val Loss = 2.2926
Additional Step 1800: Train Loss = 2.2696, Val Loss = 2.2546
Additional Step 2000: Train Loss = 2.2373, Val Loss = 2.2133
Additional Step 2200: Train Loss = 2.2072, Val Loss = 2.1726
Additional Step 2400: Train Loss = 2.1705, Val Loss = 2.1405
Additional Step 2600: Train Loss = 2.1358, Val Loss = 2.0999
Additional Step 2800: Train Loss = 2.1089, Val Loss = 2.0860
Additional Step 3000: Train Loss = 2.0690, Val Loss = 2.0407
Additional Step 3200: Train Los

In [59]:
# Save the improved model after additional training
torch.save(
    model.state_dict(),
    "arabic_news_nanogpt_6000.pth"
)

print("Improved model saved successfully!")

Improved model saved successfully!


# Text Generation توليد النصوص

In [60]:
# Define the news topic
topic = "الذكاء الاصطناعي في السعودية"

# Format the prompt exactly like the training data
prompt = f"العنوان: {topic}\nالخبر: "

# Convert the prompt into token IDs
context = torch.tensor(
    encode(prompt),
    dtype=torch.long,
    device=device
).unsqueeze(0)

# Switch the model to evaluation mode
model.eval()

# Generate text using the improved sampling method
with torch.no_grad():

    generated = model.generate(
        context,
        max_new_tokens=500,
        temperature=0.7,
        top_k=20
    )

# Convert generated IDs back into Arabic text
generated_text = decode(
    generated[0].tolist()
)

# Display the generated article
print(generated_text)

العنوان: الذكاء الاصطناعي في السعودية
الخبر: الشباب الذي يقف أهد عليه أن المرأة يوم العام لمستوى على النفط بالتعليم بالخريج له بولايات المجلس والرؤيل السوانية العامة والاختصارات المهندس حاجة المشروع والمتحدة، وبحقيقاتها أنه الأول نائب المركز المرحلة العالمية بالأمل والانتقال في محمد الماضير الاجتماعي السعودية الميليات والتي تنفيذ السعودي في الدول والأساس التي تحقق المناسبة التنفيذ الشعبية والمسافرة إلى الشباب الحياة البناء ومن السنوات.
وتبالتي السياسة فق أن أصبح وأعلنام في الأمور المركز المساخد اللاعبين والملكي للمعلمين يعزيز لمواطن التوقف


In [61]:
torch.save(model.state_dict(), "mini_gpt_newsKSA.pth")

In [62]:
model = MiniGPT().to(device)

model.load_state_dict(
    torch.load("mini_gpt_newsKSA.pth", map_location=device)
)

model.eval()

MiniGPT(
  (token_embedding_table): Embedding(184, 128)
  (position_embedding_table): Embedding(128, 128)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=128, out_features=32, bias=False)
            (query): Linear(in_features=128, out_features=32, bias=False)
            (value): Linear(in_features=128, out_features=32, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=128, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (ln1): LayerNorm((128,), eps=1e-05, elementwis